In [ ]:
# ═══════════════════════════════════════════════════
# CELL 1 — Setup
# ═══════════════════════════════════════════════════
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report, roc_auc_score,
    f1_score, precision_score, recall_score,
    accuracy_score, confusion_matrix, roc_curve
)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

print("✅ Setup complete")

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 2 — Load Temporal Data
# ═══════════════════════════════════════════════════
raw_path = Path("../data/raw/")

train_keywords = ['Monday', 'Tuesday', 'Wednesday']
test_keywords = ['Thursday', 'Friday']

csv_files = list(raw_path.glob("*.csv"))

train_dfs = []
for f in csv_files:
    if any(day in f.name for day in train_keywords):
        temp = pd.read_csv(f, low_memory=False)
        temp.columns = temp.columns.str.strip().str.replace(' ', '_').str.replace('/', '_').str.lower()
        train_dfs.append(temp)

test_dfs = []
for f in csv_files:
    if any(day in f.name for day in test_keywords):
        temp = pd.read_csv(f, low_memory=False)
        temp.columns = temp.columns.str.strip().str.replace(' ', '_').str.replace('/', '_').str.lower()
        test_dfs.append(temp)

df_train = pd.concat(train_dfs, ignore_index=True)
df_test = pd.concat(test_dfs, ignore_index=True)

df_train['label_binary'] = (df_train['label'] != 'BENIGN').astype(int)
df_test['label_binary'] = (df_test['label'] != 'BENIGN').astype(int)

print(f"Train: {len(df_train):,} ({df_train['label_binary'].sum():,} attacks)")
print(f"Test:  {len(df_test):,} ({df_test['label_binary'].sum():,} attacks)")

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 3 — Prepare Features
# ═══════════════════════════════════════════════════
exclude = ['label', 'label_binary']
train_numeric = [c for c in df_train.select_dtypes(include=[np.number]).columns 
                 if c not in exclude]
test_numeric = [c for c in df_test.select_dtypes(include=[np.number]).columns 
                if c not in exclude]
common_cols = sorted(list(set(train_numeric) & set(test_numeric)))

X_train = df_train[common_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
y_train = df_train['label_binary'].values
X_test = df_test[common_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
y_test = df_test['label_binary'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Separate benign training data for Isolation Forest
X_train_benign = X_train_scaled[y_train == 0]

print(f"Features: {len(common_cols)}")
print(f"Benign training samples: {len(X_train_benign):,}")
print("✅ Data ready")

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 4 — Train Both Models
#
# STRATEGY:
# 1. XGBoost (supervised) — catches KNOWN attack patterns
# 2. Isolation Forest (unsupervised) — catches UNKNOWN anomalies
# 3. Combine both — best of both worlds
# ═══════════════════════════════════════════════════

# ── Model 1: XGBoost (knows what attacks look like) ──
print("Training XGBoost (supervised)...")

n_benign = (y_train == 0).sum()
n_attack = (y_train == 1).sum()
scale_weight = n_benign / n_attack

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',
    use_label_encoder=False
)
xgb.fit(X_train_scaled, y_train, verbose=False)
print("  ✅ XGBoost trained")

# ── Model 2: Isolation Forest (knows what normal looks like) ──
print("\nTraining Isolation Forest (unsupervised)...")

iso = IsolationForest(
    contamination=0.05,
    n_estimators=300,
    max_samples=min(100000, len(X_train_benign)),
    random_state=42,
    n_jobs=-1
)
iso.fit(X_train_benign)
print("  ✅ Isolation Forest trained")

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 5 — Generate Individual Scores
# ═══════════════════════════════════════════════════

# XGBoost: probability of being an attack
xgb_scores = xgb.predict_proba(X_test_scaled)[:, 1]

# Isolation Forest: anomaly score (negated so higher = more anomalous)
iso_scores_raw = iso.score_samples(X_test_scaled)
# Normalize to [0, 1] range
iso_scores = (iso_scores_raw - iso_scores_raw.min()) / \
             (iso_scores_raw.max() - iso_scores_raw.min())
iso_scores = 1 - iso_scores  # Invert: higher = more anomalous

print(f"XGBoost scores:  min={xgb_scores.min():.4f}  "
      f"max={xgb_scores.max():.4f}  mean={xgb_scores.mean():.4f}")
print(f"IsoForest scores: min={iso_scores.min():.4f}  "
      f"max={iso_scores.max():.4f}  mean={iso_scores.mean():.4f}")

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 6 — Hybrid Combination Strategies
#
# We combine both models using different strategies
# to find the best balance between known and unknown
# attack detection.
# ═══════════════════════════════════════════════════

print("=" * 70)
print("HYBRID MODEL COMBINATION STRATEGIES")
print("=" * 70)

strategies = {}

# Strategy 1: Weighted Average
# Give more weight to XGBoost (it's more precise)
for xgb_weight in [0.7, 0.6, 0.5]:
    iso_weight = 1 - xgb_weight
    hybrid_scores = xgb_weight * xgb_scores + iso_weight * iso_scores
    
    # Find best threshold for this combination
    best_f1 = 0
    best_t = 0.5
    for t in np.arange(0.1, 0.9, 0.05):
        y_pred = (hybrid_scores >= t).astype(int)
        f = f1_score(y_test, y_pred, zero_division=0)
        if f > best_f1:
            best_f1 = f
            best_t = t
    
    y_pred = (hybrid_scores >= best_t).astype(int)
    
    name = f"Weighted ({xgb_weight:.0%}XGB + {iso_weight:.0%}IF)"
    strategies[name] = {
        'scores': hybrid_scores,
        'predictions': y_pred,
        'threshold': best_t,
        'f1': f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'auc': roc_auc_score(y_test, hybrid_scores)
    }

# Strategy 2: OR Logic
# Flag as attack if EITHER model flags it
for xgb_t in [0.3, 0.4, 0.5]:
    iso_threshold = np.percentile(iso_scores[y_train[: len(iso_scores)] == 0] 
                                   if len(y_train) >= len(iso_scores) 
                                   else iso_scores, 95)
    
    xgb_pred = (xgb_scores >= xgb_t).astype(int)
    iso_pred = (iso_scores >= 0.5).astype(int)
    hybrid_pred = np.maximum(xgb_pred, iso_pred)  # OR logic
    
    name = f"OR Logic (XGB≥{xgb_t}, IF≥0.5)"
    
    # Create combined score for AUC
    combined_score = np.maximum(xgb_scores, iso_scores)
    
    strategies[name] = {
        'scores': combined_score,
        'predictions': hybrid_pred,
        'threshold': f"XGB≥{xgb_t}|IF≥0.5",
        'f1': f1_score(y_test, hybrid_pred),
        'precision': precision_score(y_test, hybrid_pred),
        'recall': recall_score(y_test, hybrid_pred),
        'auc': roc_auc_score(y_test, combined_score)
    }

# Strategy 3: Max Score
# Take the maximum anomaly score from either model
hybrid_max = np.maximum(xgb_scores, iso_scores)
best_f1_max = 0
best_t_max = 0.5
for t in np.arange(0.1, 0.9, 0.05):
    y_pred = (hybrid_max >= t).astype(int)
    f = f1_score(y_test, y_pred, zero_division=0)
    if f > best_f1_max:
        best_f1_max = f
        best_t_max = t

y_pred_max = (hybrid_max >= best_t_max).astype(int)
strategies["Max Score"] = {
    'scores': hybrid_max,
    'predictions': y_pred_max,
    'threshold': best_t_max,
    'f1': f1_score(y_test, y_pred_max),
    'precision': precision_score(y_test, y_pred_max),
    'recall': recall_score(y_test, y_pred_max),
    'auc': roc_auc_score(y_test, hybrid_max)
}

# Print all strategies
print(f"\n  {'Strategy':<40} {'F1':>8} {'Precision':>10} "
      f"{'Recall':>8} {'AUC':>8}")
print(f"  {'─' * 74}")

# Add baselines
print(f"  {'XGBoost Only (t=0.5)':<40} "
      f"{f1_score(y_test, (xgb_scores>=0.5).astype(int)):>8.4f} "
      f"{precision_score(y_test, (xgb_scores>=0.5).astype(int)):>10.4f} "
      f"{recall_score(y_test, (xgb_scores>=0.5).astype(int)):>8.4f} "
      f"{roc_auc_score(y_test, xgb_scores):>8.4f}")

print(f"  {'Isolation Forest Only':<40} "
      f"{f1_score(y_test, (iso_scores>=0.5).astype(int)):>8.4f} "
      f"{precision_score(y_test, (iso_scores>=0.5).astype(int)):>10.4f} "
      f"{recall_score(y_test, (iso_scores>=0.5).astype(int)):>8.4f} "
      f"{roc_auc_score(y_test, iso_scores):>8.4f}")

print(f"  {'─' * 74}")

best_strategy = None
best_f1_overall = 0

for name, s in strategies.items():
    marker = ""
    if s['f1'] > best_f1_overall:
        best_f1_overall = s['f1']
        best_strategy = name
        marker = " ← BEST"
    
    print(f"  {name:<40} {s['f1']:>8.4f} {s['precision']:>10.4f} "
          f"{s['recall']:>8.4f} {s['auc']:>8.4f}{marker}")

print(f"\n  🏆 Best hybrid strategy: {best_strategy}")

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 7 — Best Strategy Detection by Attack Type
# ═══════════════════════════════════════════════════

best = strategies[best_strategy]

df_test_results = df_test[['label']].copy()
df_test_results['predicted'] = best['predictions']
df_test_results['score'] = best['scores']

print("=" * 70)
print(f"DETECTION BY ATTACK TYPE — {best_strategy}")
print("=" * 70)

print(f"\n  {'Attack Type':<35} {'Total':>8} {'Detected':>10} "
      f"{'Rate':>8} {'vs XGB':>10}")
print(f"  {'─' * 71}")

# XGBoost-only results for comparison
xgb_pred_baseline = (xgb_scores >= 0.5).astype(int)

for label in sorted(df_test_results['label'].unique()):
    mask = df_test_results['label'] == label
    total = mask.sum()
    
    detected_hybrid = df_test_results.loc[mask, 'predicted'].sum()
    rate_hybrid = detected_hybrid / total * 100 if total > 0 else 0
    
    detected_xgb = xgb_pred_baseline[mask.values].sum()
    rate_xgb = detected_xgb / total * 100 if total > 0 else 0
    
    diff = rate_hybrid - rate_xgb
    diff_str = f"+{diff:.1f}%" if diff > 0 else f"{diff:.1f}%"
    
    indicator = "✅" if rate_hybrid > 80 else "🟡" if rate_hybrid > 50 else "🔴"
    
    if label != 'BENIGN':
        print(f"  {indicator} {label:<33} {total:>8,} {detected_hybrid:>10,} "
              f"{rate_hybrid:>7.1f}% {diff_str:>10}")

# Benign (false alarm rate)
benign_mask = df_test_results['label'] == 'BENIGN'
fp_hybrid = df_test_results.loc[benign_mask, 'predicted'].sum()
fp_xgb = xgb_pred_baseline[benign_mask.values].sum()
print(f"\n  False alarms: {fp_hybrid:,} / {benign_mask.sum():,} "
      f"({fp_hybrid/benign_mask.sum()*100:.2f}%) "
      f"[XGB only: {fp_xgb:,}]")

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 8 — Final Visualization
# ═══════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. ROC Comparison
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_scores)
fpr_iso, tpr_iso, _ = roc_curve(y_test, iso_scores)
fpr_hybrid, tpr_hybrid, _ = roc_curve(y_test, best['scores'])

axes[0, 0].plot(fpr_xgb, tpr_xgb, color='#0F3057', lw=2,
                label=f'XGBoost (AUC={roc_auc_score(y_test, xgb_scores):.4f})')
axes[0, 0].plot(fpr_iso, tpr_iso, color='#2E86AB', lw=2,
                label=f'IsoForest (AUC={roc_auc_score(y_test, iso_scores):.4f})')
axes[0, 0].plot(fpr_hybrid, tpr_hybrid, color='#E8683F', lw=2,
                label=f'Hybrid (AUC={best["auc"]:.4f})')
axes[0, 0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0, 0].set_title('ROC Comparison', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('FPR')
axes[0, 0].set_ylabel('TPR')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Detection Rate by Attack Type
attack_labels = []
hybrid_rates = []
xgb_rates = []

for label in sorted(df_test_results['label'].unique()):
    if label == 'BENIGN':
        continue
    mask = df_test_results['label'] == label
    total = mask.sum()
    if total > 0:
        attack_labels.append(label.replace('Web Attack ', 'WA:'))
        hybrid_rates.append(
            df_test_results.loc[mask, 'predicted'].sum() / total * 100)
        xgb_rates.append(
            xgb_pred_baseline[mask.values].sum() / total * 100)

x = np.arange(len(attack_labels))
width = 0.35
axes[0, 1].bar(x - width/2, xgb_rates, width, label='XGBoost Only',
               color='#0F3057')
axes[0, 1].bar(x + width/2, hybrid_rates, width, label='Hybrid',
               color='#E8683F')
axes[0, 1].set_title('Detection Rate by Attack Type',
                      fontsize=13, fontweight='bold')
axes[0, 1].set_ylabel('Detection Rate (%)')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(attack_labels, rotation=45, ha='right', fontsize=8)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Confusion Matrix
cm = confusion_matrix(y_test, best['predictions'])
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues',
            xticklabels=['Benign', 'Attack'],
            yticklabels=['Benign', 'Attack'],
            ax=axes[1, 0])
axes[1, 0].set_title(f'Hybrid Confusion Matrix',
                      fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Actual')
axes[1, 0].set_xlabel('Predicted')

# 4. Score Distribution Comparison
axes[1, 1].hist(xgb_scores[y_test == 0], bins=100, alpha=0.3,
                label='XGB Benign', color='#2E86AB', density=True)
axes[1, 1].hist(xgb_scores[y_test == 1], bins=100, alpha=0.3,
                label='XGB Attack', color='#E8683F', density=True)
axes[1, 1].hist(iso_scores[y_test == 1], bins=100, alpha=0.3,
                label='IF Attack', color='#F6AE2D', density=True)
axes[1, 1].set_title('Score Distributions', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Anomaly Score')
axes[1, 1].legend()

plt.suptitle(f'NetSentinel — Hybrid Model Analysis ({best_strategy})',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig("../notebooks/hybrid_analysis.png", dpi=150, bbox_inches='tight')
plt.show()